# LMDeploy

A practical reference for **LMDeploy** — the LLM compression, deployment, and serving toolkit from the InternLM / OpenMMLab team. LMDeploy packages two inference engines (TurboMind and PyTorch), quantization tooling, and an OpenAI-compatible API server behind a single CLI and Python API.

> Repository: <https://github.com/InternLM/lmdeploy> · Docs: <https://lmdeploy.readthedocs.io>

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

LMDeploy is a toolkit for **compressing, deploying, and serving large language models (LLMs) and vision-language models (VLMs)**. It is maintained by the InternLM team and is the reference serving stack for the InternLM/InternVL model families, though it supports most popular open models (Llama, Qwen, Mistral, Mixtral, DeepSeek, Gemma, ChatGLM, Baichuan, and more).

### What is it?

At its core LMDeploy is built around the **TurboMind** inference engine — a C++/CUDA engine derived from NVIDIA's FasterTransformer — plus a pure-PyTorch fallback engine. On top of those engines it ships:

- a Python `pipeline` API for batch/offline inference,
- an OpenAI-compatible **`api_server`** for online serving,
- quantization workflows (AWQ W4A16, online/offline KV-cache INT4/INT8, W8A8 SmoothQuant),
- a CLI (`lmdeploy chat`, `lmdeploy serve`, `lmdeploy lite`, `lmdeploy convert`).

### Why use it?

- **Throughput.** TurboMind's persistent (continuous) batching and blocked KV cache deliver some of the highest request throughput among open serving stacks — frequently quoted as ~1.8x vs. vLLM on comparable hardware for InternLM-class models.
- **First-class quantization.** 4-bit (AWQ) weight quantization and KV-cache quantization are integrated and benchmarked, not bolted on.
- **Low latency.** Hand-tuned CUDA kernels and a C++ runtime keep per-token latency low.
- **Drop-in API.** The `api_server` speaks the OpenAI `/v1/chat/completions` and `/v1/completions` schema, so existing OpenAI SDK clients work unchanged.

### When to use it?

- You serve InternLM, InternVL, Qwen, or Llama-family models and want maximum tokens/sec per GPU.
- You need 4-bit weight + quantized KV cache to fit a large model on limited VRAM.
- You want an OpenAI-compatible endpoint without writing a serving layer yourself.
- You are serving vision-language models and want a single stack for text and multimodal.

## Key Features

### Core Capabilities of LMDeploy

| Feature | Description | Benefit |
|---------|-------------|---------|
| TurboMind engine | C++/CUDA engine with persistent batching, blocked KV cache, dynamic split-and-fuse, tensor parallelism | Highest throughput / lowest latency path |
| PyTorch engine | Pure-Python engine that runs models without a custom kernel port | Broadest model coverage; easy to extend |
| Persistent (continuous) batching | Requests join/leave the running batch token-by-token instead of waiting for the batch to drain | High GPU utilization under mixed-length traffic |
| Blocked KV cache (paged attention) | KV cache split into fixed-size blocks, allocated on demand | Less fragmentation, larger effective batch |
| Weight-only quantization (AWQ W4A16) | 4-bit weights, FP16 activations via `lmdeploy lite auto_awq` | ~4x smaller weights, faster decode |
| KV-cache quantization | Online INT4/INT8 KV cache (`quant_policy=4` or `8`) | Fits longer contexts / bigger batches in VRAM |
| OpenAI-compatible server | `lmdeploy serve api_server` exposes `/v1/chat/completions` etc. | Reuse OpenAI SDK and tooling |
| Tensor parallelism | `--tp N` shards a model across N GPUs | Serve models larger than one GPU |
| VLM support | Serves InternVL, LLaVA, Qwen-VL, etc. through the same pipeline/server | One stack for text + multimodal |

## Architecture Overview

LMDeploy exposes a thin user surface (CLI / `pipeline` / `api_server`) over a pluggable engine backend.

```
        Clients                       LMDeploy surface                  Engine backend
 ┌──────────────────┐         ┌──────────────────────────────┐   ┌────────────────────────┐
 │ OpenAI SDK / curl│──HTTP──▶│ api_server (FastAPI/uvicorn) │   │ TurboMind (C++/CUDA)   │
 │ Python script    │──call──▶│ pipeline() Python API        │──▶│  • persistent batching │
 │ terminal         │──CLI───▶│ lmdeploy chat / serve / lite │   │  • blocked KV cache    │
 └──────────────────┘         └──────────────────────────────┘   │  • tensor parallel     │
                                            │                     ├────────────────────────┤
                                            └────── or ──────────▶│ PyTorch engine         │
                                                                  │  • eager / graph mode  │
                                                                  └────────────────────────┘
```

### Components

1. **Surface layer** — the CLI, the `lmdeploy.pipeline()` Python API, and the `api_server`. All three resolve to an engine config and a model.
2. **TurboMind engine** — the default high-performance backend. Converts HF weights into TurboMind format (online or via `lmdeploy convert`) and runs persistent batching with a blocked KV cache. Configured through `TurbomindEngineConfig`.
3. **PyTorch engine** — a Python-native backend (`PytorchEngineConfig`) for models without a TurboMind port or for rapid iteration; supports many of the same scheduling features.
4. **Quantization toolkit (`lmdeploy lite`)** — AWQ calibration and weight packing, plus KV-cache quant policies applied at runtime.

## Installation

### Prerequisites

- **Linux** with an NVIDIA GPU (CUDA 11.8 or 12.x). TurboMind requires a CUDA GPU; the PyTorch engine can run on CPU but is slow.
- **Python 3.8–3.12**.
- A matching **PyTorch** build (installed automatically as a dependency, but pin CUDA-matched wheels for reproducibility).
- ~16 GB+ VRAM for a 7B model in FP16; far less with W4A16.

### Installation Steps

LMDeploy ships prebuilt CUDA wheels on PyPI, so a plain `pip install` is normally enough.

**Note**: Uncomment the cell below to install (e.g. in Google Colab with a GPU runtime).

In [ ]:
# Uncomment to install (Linux + NVIDIA GPU):
# !pip install lmdeploy

# CUDA 11.8 build (if your environment needs the cu118 wheel):
# !pip install lmdeploy --extra-index-url https://download.pytorch.org/whl/cu118

# Verify the install and see the CLI surface:
# !lmdeploy --version
# !lmdeploy --help

## Basic Usage

### Quick Start Example

There are three entry points. The fastest way to validate a model is the **offline `pipeline` API**, which batches a list of prompts and returns completions. Below, `model_path` can be an HF Hub id (downloaded on first use) or a local directory.

In [ ]:
# Offline batch inference with the pipeline API.
# Runs on a CUDA GPU; the first call downloads the model from the HF Hub.
from lmdeploy import pipeline, TurbomindEngineConfig, GenerationConfig

model_path = "internlm/internlm2_5-7b-chat"  # any HF id or local path

pipe = pipeline(
    model_path,
    backend_config=TurbomindEngineConfig(
        session_len=8192,      # max context length
        cache_max_entry_count=0.8,  # fraction of free VRAM for the KV cache
    ),
)

prompts = [
    "Explain continuous batching in one sentence.",
    "Write a haiku about GPU memory.",
]
gen = GenerationConfig(temperature=0.7, top_p=0.9, max_new_tokens=256)

responses = pipe(prompts, gen_config=gen)
for r in responses:
    print(r.text)
    print("-" * 40)


You can also chat interactively from the terminal, or serve the model as an OpenAI-compatible HTTP endpoint:

```bash
# Interactive REPL in the terminal (TurboMind engine by default):
lmdeploy chat internlm/internlm2_5-7b-chat

# Launch an OpenAI-compatible server on :23333
lmdeploy serve api_server internlm/internlm2_5-7b-chat \
    --server-port 23333 \
    --tp 1 \
    --cache-max-entry-count 0.8
```

Once the server is up, any OpenAI client works against it — note the `/v1` base URL and that the API key is arbitrary (auth is off by default).

In [ ]:
# Call the LMDeploy api_server with the OpenAI Python SDK.
# Assumes `lmdeploy serve api_server ... --server-port 23333` is running.
from openai import OpenAI

client = OpenAI(api_key="EMPTY", base_url="http://localhost:23333/v1")

# Model id == the name the server reports; query it if unsure:
model_id = client.models.list().data[0].id

resp = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "What problem does a blocked KV cache solve?"},
    ],
    temperature=0.6,
    max_tokens=200,
    stream=False,
)
print(resp.choices[0].message.content)


## Advanced Features

### Quantization, multi-GPU, and the PyTorch engine

#### 4-bit weight quantization (AWQ, W4A16)

`lmdeploy lite auto_awq` calibrates and packs weights to 4-bit. The quantized checkpoint is then served exactly like an FP16 one — pass `model_format="awq"` so TurboMind loads the packed weights.

#### KV-cache quantization

Set `quant_policy=8` (INT8) or `quant_policy=4` (INT4) on the engine config to quantize the KV cache at runtime. This roughly doubles or quadruples the context/batch that fits in a given amount of VRAM, with minimal quality loss.

#### Tensor parallelism

`tp=N` (or `--tp N` on the CLI) shards attention and MLP weights across `N` GPUs to serve models too large for one card. `N` must divide the model's attention head count.

In [ ]:
# (a) Quantize to 4-bit AWQ from the CLI:
# !lmdeploy lite auto_awq internlm/internlm2_5-7b-chat \
#       --calib-dataset ptb --calib-samples 128 --batch-size 1 \
#       --work-dir ./internlm2_5-7b-chat-4bit

# (b) Serve the quantized model with an INT8 KV cache across 2 GPUs:
from lmdeploy import pipeline, TurbomindEngineConfig, GenerationConfig

pipe = pipeline(
    "./internlm2_5-7b-chat-4bit",
    backend_config=TurbomindEngineConfig(
        model_format="awq",        # load 4-bit packed weights
        quant_policy=8,            # INT8 KV cache (use 4 for INT4)
        tp=2,                      # shard across 2 GPUs
        cache_max_entry_count=0.85,
        session_len=16384,
    ),
)
print(pipe(["Summarize AWQ in one line."],
           gen_config=GenerationConfig(max_new_tokens=64))[0].text)

# (c) Use the PyTorch engine instead of TurboMind (broader model coverage):
from lmdeploy import PytorchEngineConfig
pipe_pt = pipeline(
    "internlm/internlm2_5-7b-chat",
    backend_config=PytorchEngineConfig(tp=1, session_len=8192),
)


## Use Cases

### Real-world Applications of LMDeploy

#### Use Case 1: High-throughput chat API behind an existing OpenAI integration

- **Context**: A product already calls the OpenAI Python SDK and wants to swap in a self-hosted open model to cut cost.
- **Implementation**: Run `lmdeploy serve api_server <model> --server-port 23333`, point the SDK's `base_url` at it. No client code changes.
- **Results**: OpenAI-compatible streaming endpoint with TurboMind throughput; cost shifts from per-token billing to fixed GPU rental.

#### Use Case 2: Fitting a large model on a single 24 GB GPU

- **Context**: A 14B–20B model won't fit in FP16 on one consumer/L4-class GPU.
- **Implementation**: `lmdeploy lite auto_awq` to W4A16, then serve with `model_format="awq"` and `quant_policy=4/8` for the KV cache.
- **Results**: ~4x smaller weights plus a quantized KV cache let the model and a usable context window fit, often with higher decode throughput.

#### Use Case 3: Offline batch generation / dataset labeling

- **Context**: Generating synthetic data or scoring a large corpus, where latency per request doesn't matter but total throughput does.
- **Implementation**: Use the `pipeline` API and pass the full list of prompts; persistent batching keeps the GPU saturated.
- **Results**: Maximal tokens/sec; no HTTP overhead.

## Best Practices

### Recommended Practices for LMDeploy

1. **Tune `cache_max_entry_count` deliberately.** It's the *fraction of free VRAM* given to the KV cache (default 0.8). Higher → larger batches/contexts but a greater OOM risk if other processes share the GPU. Lower it on shared cards.
2. **Match `session_len` to real traffic.** Setting it far above the longest prompt+completion wastes KV-cache capacity; setting it too low truncates requests.
3. **Quantize for memory-bound decode.** W4A16 (AWQ) plus an INT8 KV cache is the standard recipe for fitting bigger models and speeding token generation; benchmark quality on your task first.
4. **Prefer TurboMind when your model is supported; fall back to the PyTorch engine otherwise.** TurboMind is faster; the PyTorch engine covers more architectures and is easier to debug.
5. **Pin versions and CUDA wheels.** TurboMind kernels are tied to specific CUDA/PyTorch builds — pin `lmdeploy`, `torch`, and the CUDA wheel index in your image for reproducible deployments.

## Common Pitfalls

### What to Avoid When Using LMDeploy

1. **Forgetting `model_format="awq"` on quantized weights.** Loading an AWQ checkpoint without it makes TurboMind misread the packed weights and produce garbage or errors. Always set the format to match the checkpoint.
2. **Over-subscribing VRAM with `cache_max_entry_count`.** A value near 1.0 on a GPU shared with other jobs causes OOM mid-serving. Leave headroom and account for other processes.
3. **Choosing an invalid `tp`.** Tensor-parallel size must evenly divide the number of attention heads (and the GPU count). A mismatched `--tp` fails at load time.
4. **Assuming every HF model has a TurboMind port.** Unsupported architectures must use the PyTorch engine (`PytorchEngineConfig`); check the supported-models list before standardizing on TurboMind.

## Performance Optimization

### Optimizing LMDeploy for Production

#### Configuration Tuning

Key parameters to optimize:

- **`cache_max_entry_count`** — fraction of free VRAM for the blocked KV cache. The single biggest lever on max batch size and throughput.
- **`quant_policy`** — `0` (FP16 KV), `8` (INT8), or `4` (INT4). Quantizing the KV cache frees VRAM for larger batches/longer contexts.
- **`max_batch_size`** — caps concurrent sequences in the persistent batch; raise it to push throughput until you hit the KV-cache or latency ceiling.
- **`tp`** — tensor-parallel degree; scales a single model across GPUs to raise aggregate throughput and fit larger models.
- **`session_len`** — sized to actual prompt+completion lengths so KV-cache budget isn't wasted.

In [ ]:
# Micro-benchmark: measure decode throughput (tokens/sec) for a config.
import time
from lmdeploy import pipeline, TurbomindEngineConfig, GenerationConfig

pipe = pipeline(
    "internlm/internlm2_5-7b-chat",
    backend_config=TurbomindEngineConfig(
        cache_max_entry_count=0.8,
        max_batch_size=128,
        quant_policy=8,
    ),
)

prompts = ["Write a 200-word explanation of paged attention."] * 32
gen = GenerationConfig(max_new_tokens=256, temperature=0.0)

t0 = time.perf_counter()
outs = pipe(prompts, gen_config=gen)
dt = time.perf_counter() - t0

total_tokens = sum(o.generate_token_len for o in outs)
print(f"requests={len(prompts)}  total_new_tokens={total_tokens}")
print(f"wall={dt:.2f}s  throughput={total_tokens / dt:,.0f} tok/s")

# LMDeploy also ships standalone benchmark scripts in benchmark/:
#   python benchmark/profile_throughput.py <dataset> <model> --backend turbomind


## Production Deployment

### Deploying LMDeploy in Production

#### Docker Deployment

LMDeploy publishes official CUDA images (`openmmlab/lmdeploy`). Mount your model cache and expose the server port.

```dockerfile
FROM openmmlab/lmdeploy:latest

# (Optional) bake a model in, or mount it at runtime via -v
ENV HF_HOME=/root/.cache/huggingface
EXPOSE 23333

ENTRYPOINT ["lmdeploy", "serve", "api_server", \
            "internlm/internlm2_5-7b-chat", \
            "--server-name", "0.0.0.0", \
            "--server-port", "23333", \
            "--cache-max-entry-count", "0.8"]
```

```bash
docker run --runtime nvidia --gpus all \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    -p 23333:23333 --ipc=host \
    openmmlab/lmdeploy:latest \
    lmdeploy serve api_server internlm/internlm2_5-7b-chat --server-port 23333
```

#### Kubernetes Deployment

Request a GPU via `nvidia.com/gpu`, and add readiness/liveness probes on the server's `/health` endpoint.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: lmdeploy-internlm
spec:
  replicas: 1
  selector:
    matchLabels: { app: lmdeploy-internlm }
  template:
    metadata:
      labels: { app: lmdeploy-internlm }
    spec:
      containers:
        - name: lmdeploy
          image: openmmlab/lmdeploy:latest
          args:
            - lmdeploy
            - serve
            - api_server
            - internlm/internlm2_5-7b-chat
            - --server-name=0.0.0.0
            - --server-port=23333
            - --cache-max-entry-count=0.8
          ports:
            - containerPort: 23333
          resources:
            limits:
              nvidia.com/gpu: 1
          readinessProbe:
            httpGet: { path: /health, port: 23333 }
            initialDelaySeconds: 60
            periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: lmdeploy-internlm
spec:
  selector: { app: lmdeploy-internlm }
  ports:
    - port: 80
      targetPort: 23333
```

## Monitoring and Observability

### Monitoring LMDeploy in Production

#### Key Metrics to Track

- **Throughput (tokens/sec)** — aggregate generated tokens per second; the headline capacity metric.
- **Latency** — time-to-first-token (TTFT) and inter-token latency (ITL); these drive perceived responsiveness for streaming.
- **GPU memory & KV-cache occupancy** — how full the blocked KV cache is; sustained saturation means you're batch-limited and should consider quantization or more GPUs.
- **Request queue depth & rejection rate** — requests waiting to enter the persistent batch; rising depth signals overload.
- **GPU utilization (`nvidia-smi` / DCGM)** — low utilization under load points to a CPU/IO bottleneck rather than compute.

#### Logging Best Practices

- Run the server with `--log-level INFO` (or `DEBUG` when diagnosing) and ship stdout/stderr to your log pipeline.
- Scrape GPU metrics with NVIDIA **DCGM-Exporter** into Prometheus and graph them in Grafana alongside request metrics.
- Put the OpenAI-compatible endpoint behind a gateway (e.g. NGINX, Envoy) that records request counts, status codes, and latency percentiles.

## Troubleshooting

### Common Issues with LMDeploy

#### Issue 1: CUDA out-of-memory at startup or under load

**Symptoms**: `CUDA out of memory` when loading the model or once concurrency rises.

**Cause**: `cache_max_entry_count` reserves too much VRAM, the model is too large for the GPU, or another process shares the card.

**Solution**: Lower `cache_max_entry_count` (e.g. 0.5), enable KV-cache quantization (`quant_policy=8/4`), quantize weights to W4A16, or increase `tp` to shard across more GPUs.

#### Issue 2: Quantized model outputs garbage

**Symptoms**: An AWQ checkpoint produces gibberish or NaNs.

**Cause**: `model_format` not set to `"awq"`, so TurboMind misinterprets the packed 4-bit weights.

**Solution**: Pass `model_format="awq"` in the engine config (or `--model-format awq` on the CLI), and confirm the checkpoint was produced by `lmdeploy lite auto_awq`.

#### Issue 3: Model fails to load on the TurboMind engine

**Symptoms**: An unsupported-architecture or conversion error when starting.

**Cause**: The model has no TurboMind port, or its config isn't recognized.

**Solution**: Switch to the PyTorch engine via `PytorchEngineConfig`, or check the supported-models list and update LMDeploy to a version that supports the architecture.

## Comparison with Alternatives

### How LMDeploy Compares to Other Solutions

| Feature | LMDeploy | vLLM | TGI (Text Generation Inference) |
|---------|----------|------|---------------------------------|
| Core engine | TurboMind (C++/CUDA) + PyTorch | Python + custom CUDA (PagedAttention) | Rust server + PyTorch/CUDA |
| Continuous batching | Yes (persistent batching) | Yes | Yes |
| Paged/blocked KV cache | Yes | Yes (PagedAttention) | Yes |
| Weight quantization | AWQ W4A16, W8A8 SmoothQuant | AWQ, GPTQ, FP8, others | AWQ, GPTQ, EETQ, others |
| KV-cache quantization | INT4 / INT8 (built-in) | FP8 KV cache | FP8 KV cache |
| OpenAI-compatible API | Yes | Yes | Yes (and TGI-native) |
| Standout strength | Top throughput on InternLM/Qwen/Llama; integrated quantization | Huge model/feature coverage, large community | Tight HF Hub integration, production hardening |

### When to Choose This Tool

Choose LMDeploy when:

- You want maximum throughput on InternLM/InternVL, Qwen, or Llama-family models.
- Integrated 4-bit weight + quantized KV cache to fit large models on small GPUs is a priority.
- You need an OpenAI-compatible server with minimal setup and strong default performance.

## Resources

### Official Documentation

- Official docs: <https://lmdeploy.readthedocs.io>
- GitHub repository: <https://github.com/InternLM/lmdeploy>
- Supported models list: <https://lmdeploy.readthedocs.io/en/latest/supported_models/supported_models.html>

### Tutorials and Guides

- Quantization (AWQ / W4A16) guide: <https://lmdeploy.readthedocs.io/en/latest/quantization/w4a16.html>
- KV-cache quantization: <https://lmdeploy.readthedocs.io/en/latest/quantization/kv_quant.html>
- OpenAI-compatible `api_server`: <https://lmdeploy.readthedocs.io/en/latest/serving/api_server.html>

### Community Resources

- GitHub Issues & Discussions: <https://github.com/InternLM/lmdeploy/issues>
- InternLM organization: <https://github.com/InternLM>
- PyPI package: <https://pypi.org/project/lmdeploy/>

### Related Technologies

- **TurboMind / FasterTransformer** — the CUDA inference lineage behind LMDeploy's fast engine.
- **AWQ (Activation-aware Weight Quantization)** — the 4-bit method LMDeploy uses for W4A16.
- **vLLM, TGI, TensorRT-LLM, SGLang** — alternative high-throughput LLM serving stacks.